<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/Filter_Bed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Filter BED notebook

short script to extend boundaries of a GTF file to arbitrary size and intersect it with user BED files stored into GitHub repo.

## Mount Google Drive

No more required.

In [0]:
# from google.colab import drive
# drive.mount('/content/drive')

## install dependency, download data and annotation

In [18]:
! rm -rf Tutorials
! git clone https://github.com/andreagrioni/Tutorials.git
! apt-get install bedtools
! mkdir results
! unzip ./Tutorials/data/filter_bed/refseq_grch38.zip

Cloning into 'Tutorials'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 143 (delta 3), reused 0 (delta 0), pack-reused 132
Receiving objects: 100% (143/143), 7.04 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
bedtools is already the newest version (2.26.0+dfsg-5).
The following package was automatically installed and is no longer required:
  libnvidia-common-430
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
mkdir: cannot create directory ‘results’: File exists
Archive:  ./Tutorials/data/filter_bed/refseq_grch38.zip
  inflating: refseq_grch38.tsv       


In [19]:
! ls Tutorials/data/filter_bed

hotspots.unstranded.threshold.0.15.bed	hotspots.unstranded.threshold.0.65.bed
hotspots.unstranded.threshold.0.1.bed	hotspots.unstranded.threshold.0.6.bed
hotspots.unstranded.threshold.0.25.bed	hotspots.unstranded.threshold.0.75.bed
hotspots.unstranded.threshold.0.2.bed	hotspots.unstranded.threshold.0.7.bed
hotspots.unstranded.threshold.0.35.bed	hotspots.unstranded.threshold.0.85.bed
hotspots.unstranded.threshold.0.3.bed	hotspots.unstranded.threshold.0.8.bed
hotspots.unstranded.threshold.0.45.bed	hotspots.unstranded.threshold.0.95.bed
hotspots.unstranded.threshold.0.4.bed	hotspots.unstranded.threshold.0.9.bed
hotspots.unstranded.threshold.0.55.bed	README.md
hotspots.unstranded.threshold.0.5.bed	refseq_grch38.zip


## Extend TSS

from: https://randomstate.net/2018-06-28-getting-refseq-gene-tss-from-ucsc/

Extend TSS end coordinates of 1000bp inside gene.

txStart and txEnd indicate where the transcript starts or ends respectively. If it is from the + strand, then txStart is the TSS; if it is from the – strand, then txEnd is the TSS. Remember in a bed format, the start coordinates are 0-based, and the end coordinates are 1-based.


In [0]:
def resize_annotation(infile, extend=1000, strand=True, outfile="refseq_hg38_UCSC_TSS.bed"):
  with open(infile, 'r') as f, open(outfile, "w") as output:
    for index, line in enumerate(f.readlines()):
      if index == 0:
        continue
      column = line.strip().split("\t")
      if strand:
        if column[3] == "+":
          start = column[4]
          end = int(column[4]) + extend
        elif column[3] == "-":
          start = int(column[5]) - extend
          end = int(column[5])
          if start < 0:
            start = 0
      else:
        # ToDo
        pass

      string_out = f'{column[2]}\t{start}\t{end}\t{column[1]}_{column[12]}\t.\t{column[3]}\n'
      output.write(string_out)
  return outfile

## Intersect with BedTools

In [0]:
import subprocess


def run_bedtools(candidate_file, annotation, outdir):

  candidate_file_name = os.path.basename(candidate_file).replace('.bed', '.filtered.tss_1kb.bed')
  outname = f'{outdir}/{candidate_file_name}'
  cmd = f'bedtools intersect -a {candidate_file} -b {annotation} -u > {outname}'
  print(cmd)
  try:
    subprocess.run(cmd, shell=True)
  except:
    return "error"

## run code

In [30]:
import os

annotation_path = "refseq_grch38.tsv"
annotation_resize = resize_annotation( annotation_path, extend=1000)

bed_dir_path = "Tutorials/data/filter_bed/"
output_dir_path = "results/"

for bed_file in os.listdir(bed_dir_path):
  if not bed_file.endswith('.bed'):
    continue
  candidates_bed_path = os.path.join(bed_dir_path, bed_file)
  print(candidates_bed_path)
  run_bedtools(candidates_bed_path, annotation_resize, output_dir_path)


Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.85.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.85.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.85.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.95.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.95.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.95.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.65.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.65.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.65.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.9.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.9.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.9.filtered.tss_1kb.bed
Tut

In [31]:
! more results//hotspots.unstranded.threshold.0.85.filtered.tss_1kb.bed | head -n 10

chr1	32335739	32335849	prediction_chr1_1	0.985584795475006	.
chr1	34723157	34723257	prediction_chr1_2	0.903949618339539	.
chr1	65058399	65058544	prediction_chr1_14	0.999999761581421	.
chr1	65066849	65066979	prediction_chr1_15	0.954982221126556	.
chr1	227941576	227941686	prediction_chr1_19	0.987961113452911	.
chr11	62653198	62653298	prediction_chr11_0	0.902726709842682	.
chr11	126268803	126268903	prediction_chr11_2	0.880615174770355	.
chr14	94227448	94227548	prediction_chr14_1	9.271241426467895508e-01	.
chr16	87834568	87834688	prediction_chr16_0	0.968097984790802	.
chr17	49361501	49361601	prediction_chr17_1	0.876615881919861	.


## Download Results

This section allows the user to download results

In [33]:
from google.colab import files
from zipfile import ZipFile


with ZipFile('./results_tss_filtered.zip', 'w') as zipObj:
  for bed_file in os.listdir(output_dir_path):
    bed_path = os.path.join(output_dir_path, bed_file)
    print(bed_path)
    zipObj.write(bed_path)

files.download('./results_tss_filtered.zip')
files.download('./refseq_hg38_UCSC_TSS.bed')

results/hotspots.unstranded.threshold.0.95.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.7.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.65.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.4.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.15.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.8.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.75.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.2.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.35.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.55.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.25.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.1.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.85.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.3.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.6.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.5.filte